In [1]:
# ==========================================
# PHASE 2 : PSEUDO LABEL GENERATION
# Support Integrity Auditor (SIA)
# ==========================================

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv("processed_tickets.csv")

print("Dataset Shape:", df.shape)

# ==========================================
# PRIORITY MAP
# ==========================================

priority_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

df["priority_numeric"] = (
    df["Priority_Level"]
    .map(priority_map)
)

# ==========================================
# SIGNAL 1 : KEYWORD SEVERITY
# ==========================================

severity_keywords = {

    "critical": [

        "server down",
        "system crash",
        "security breach",
        "payment failed",
        "account locked",
        "unable to login",
        "data loss",
        "website down",
        "outage",
        "unauthorized access"

    ],

    "high": [

        "error",
        "failed",
        "issue",
        "problem",
        "slow",
        "bug",
        "delay",
        "exception"

    ]
}


def keyword_score(text):

    text = str(text).lower()

    score = 0

    for word in severity_keywords["critical"]:
        if word in text:
            score += 4

    for word in severity_keywords["high"]:
        if word in text:
            score += 2

    return score


df["keyword_score"] = (
    df["clean_text"]
    .apply(keyword_score)
)

# ==========================================
# SIGNAL 2 : RESOLUTION TIME
# ==========================================

def resolution_score(hours):

    if hours <= 11:
        return 1

    elif hours <= 27:
        return 2

    elif hours <= 58:
        return 3

    else:
        return 4


df["resolution_score"] = (
    df["Resolution_Time_Hours"]
    .apply(resolution_score)
)

# ==========================================
# SIGNAL 3 : SATISFACTION SCORE
# ==========================================

def satisfaction_signal(score):

    if score == 1:
        return 4

    elif score == 2:
        return 3

    elif score == 3:
        return 2

    else:
        return 1


df["satisfaction_signal"] = (
    df["Satisfaction_Score"]
    .apply(satisfaction_signal)
)

# ==========================================
# SIGNAL 4 : CATEGORY SEVERITY
# ==========================================

category_severity = (
    df.groupby("Issue_Category")
    ["priority_numeric"]
    .mean()
)

category_dict = (
    category_severity
    .round()
    .to_dict()
)

df["category_signal"] = (
    df["Issue_Category"]
    .map(category_dict)
)

# ==========================================
# SIGNAL 5 : EMBEDDING SEVERITY
# ==========================================

print("\nGenerating Sentence Embeddings...")

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model.encode(
    df["clean_text"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

print("\nRunning KMeans Clustering...")

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(
    embeddings
)

df["embedding_cluster"] = clusters

# ==========================================
# CLUSTER -> SEVERITY
# ==========================================

cluster_priority = (
    df.groupby("embedding_cluster")
    ["priority_numeric"]
    .mean()
)

cluster_map = {}

for cluster, value in cluster_priority.items():

    if value < 1.5:
        cluster_map[cluster] = 1

    elif value < 2.5:
        cluster_map[cluster] = 2

    elif value < 3.5:
        cluster_map[cluster] = 3

    else:
        cluster_map[cluster] = 4

df["embedding_signal"] = (
    df["embedding_cluster"]
    .map(cluster_map)
)

# ==========================================
# SEVERITY FUSION
# ==========================================

df["severity_score"] = (

    0.30 * df["keyword_score"]

    +

    0.25 * df["resolution_score"]

    +

    0.15 * df["satisfaction_signal"]

    +

    0.15 * df["category_signal"]

    +

    0.15 * df["embedding_signal"]

)

# ==========================================
# INFERRED SEVERITY
# ==========================================

def severity_class(score):

    if score < 2:
        return "Low"

    elif score < 4:
        return "Medium"

    elif score < 6:
        return "High"

    else:
        return "Critical"


df["inferred_severity"] = (
    df["severity_score"]
    .apply(severity_class)
)

# ==========================================
# NUMERIC INFERRED SEVERITY
# ==========================================

severity_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

df["inferred_numeric"] = (
    df["inferred_severity"]
    .map(severity_map)
)

# ==========================================
# MISMATCH LABEL
# ==========================================

df["mismatch"] = (

    df["priority_numeric"]

    !=

    df["inferred_numeric"]

).astype(int)

# ==========================================
# MISMATCH TYPE
# ==========================================

def mismatch_type(row):

    if row["inferred_numeric"] > row["priority_numeric"]:
        return "Hidden Crisis"

    elif row["inferred_numeric"] < row["priority_numeric"]:
        return "False Alarm"

    else:
        return "Consistent"


df["mismatch_type"] = (
    df.apply(
        mismatch_type,
        axis=1
    )
)

# ==========================================
# SUMMARY
# ==========================================

print("\n==============================")
print("INFERRED SEVERITY")
print("==============================")
print(df["inferred_severity"].value_counts())

print("\n==============================")
print("MISMATCH LABELS")
print("==============================")
print(df["mismatch"].value_counts())

print("\n==============================")
print("MISMATCH TYPES")
print("==============================")
print(df["mismatch_type"].value_counts())

# ==========================================
# SAVE OUTPUT
# ==========================================

df.to_csv(
    "pseudo_labeled_tickets.csv",
    index=False
)

print(
    "\nSaved: pseudo_labeled_tickets.csv"
)

Dataset Shape: (20000, 20)

Generating Sentence Embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]


Running KMeans Clustering...

INFERRED SEVERITY
inferred_severity
Low       15868
Medium     4128
High          4
Name: count, dtype: int64

MISMATCH LABELS
mismatch
1    12009
0     7991
Name: count, dtype: int64

MISMATCH TYPES
mismatch_type
False Alarm      10520
Consistent        7991
Hidden Crisis     1489
Name: count, dtype: int64

Saved: pseudo_labeled_tickets.csv


In [2]:
print(df["mismatch"].value_counts())
print(df["inferred_severity"].value_counts())

mismatch
1    12009
0     7991
Name: count, dtype: int64
inferred_severity
Low       15868
Medium     4128
High          4
Name: count, dtype: int64
